# 🎮 Part 7: Hands-On Code Activities

**Image Classification with CNNs — Week 24**  
**Python Machine Learning Course — Learn and Help**

This notebook contains the three hands-on coding activities from the CNN lesson. Run each activity to see CNNs in action!

---

## Activity 1: Build a CNN in Keras (MNIST Digits)

Let's classify the same MNIST digits from Week 23 — but this time with a CNN instead of a regular neural network.

In [ ]:
# ===========================================
# CNN for MNIST Digit Classification (Keras)
# ===========================================
# Compare this to Week 23's regular neural network!

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import time

# Step 1: Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Step 2: Prepare data — CNN needs 4D input: (samples, height, width, channels)
# Notice: we DON'T flatten to 784! We keep the 28x28 shape!
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# Step 3: Build the CNN
model = keras.Sequential([
    # --- Convolution Block 1 ---
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    #   32 filters, each 3x3 pixels, using ReLU activation
    layers.MaxPooling2D((2, 2)),
    #   Max pooling with 2x2 window — image shrinks from 26x26 to 13x13

    # --- Convolution Block 2 ---
    layers.Conv2D(64, (3, 3), activation="relu"),
    #   64 filters, each 3x3 — finds more complex patterns
    layers.MaxPooling2D((2, 2)),
    #   Shrinks again

    # --- Flatten + Dense Layers (same as Week 23 from here) ---
    layers.Flatten(),
    #   Convert 2D feature maps to 1D list
    layers.Dense(64, activation="relu"),
    #   Regular hidden layer
    layers.Dense(10, activation="softmax")
    #   Output: 10 neurons for digits 0-9
])

# Step 4: Show what we built
model.summary()

# Step 5: Compile and train
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

print("\n Training CNN...")
print("=" * 50)
start = time.time()
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)
cnn_time = time.time() - start

# Step 6: Test
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"\n{'=' * 50}")
print(f"CNN Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Training Time: {cnn_time:.1f}s")
print(f"\nCompare to Week 23:")
print(f"   Regular NN (Keras):     ~97.5%")
print(f"   CNN (this week):        {test_accuracy * 100:.2f}%")

### Key differences from Week 23's code:

* We use `Conv2D` and `MaxPooling2D` layers instead of only `Dense` layers
* We do NOT flatten the image at the start — we keep it as 28x28
* We reshape to `(28, 28, 1)` — the `1` means one color channel (grayscale)
* We still use `Flatten()` + `Dense()` at the end, just like a regular NN

---

## Activity 2: Fashion-MNIST — Classify Clothing Items

MNIST digits are "too easy" — let's try something harder! **Fashion-MNIST** has the same format (28x28 grayscale) but contains clothing items instead of digits.

In [ ]:
# ===========================================
# CNN for Fashion-MNIST (Clothing Classification)
# ===========================================
# Same format as MNIST, but much harder!

import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# Step 1: Load Fashion-MNIST (built into Keras!)
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Class names (Fashion-MNIST uses numbers 0-9 for these categories)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Step 2: Prepare data
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# Step 3: Build a slightly deeper CNN
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),  # Extra conv layer!
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

# Step 4: Train
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

print("Training CNN on Fashion-MNIST...")
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

# Step 5: Test
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nFashion-MNIST Accuracy: {test_acc * 100:.2f}%")

# Step 6: Show predictions with class names
predictions = model.predict(x_test[:10])
print("\nFirst 10 predictions:")
for i in range(10):
    pred_class = np.argmax(predictions[i])
    confidence = predictions[i][pred_class] * 100
    actual = y_test[i]
    status = "CORRECT" if pred_class == actual else "WRONG"
    print(f"  [{status}] Predicted: {class_names[pred_class]:12s} | Actual: {class_names[actual]:12s} | Confidence: {confidence:.1f}%")

---

## Activity 3: The Ultimate Week 23 vs Week 24 Showdown

Run this script to compare all approaches on the same dataset:

In [ ]:
# ===========================================
# SHOWDOWN: Regular NN vs CNN vs Random Forest
# ===========================================

import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
from tensorflow.keras import layers
import time

# Load data
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_flat = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test_flat = x_test.reshape(-1, 784).astype("float32") / 255.0
x_train_cnn = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test_cnn = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

results = {}

# 1. Random Forest
print("Training Random Forest...")
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(x_train_flat[:10000], y_train[:10000])
results['Random Forest'] = (accuracy_score(y_test, rf.predict(x_test_flat)), time.time()-start, 'Classic ML')
print(f"   Done! {results['Random Forest'][0]*100:.2f}%")

# 2. Regular NN (scikit-learn)
print("\nTraining Regular NN (scikit-learn)...")
start = time.time()
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=42, verbose=False)
mlp.fit(x_train_flat, y_train)
results['Regular NN (sklearn)'] = (accuracy_score(y_test, mlp.predict(x_test_flat)), time.time()-start, 'Neural Net')
print(f"   Done! {results['Regular NN (sklearn)'][0]*100:.2f}%")

# 3. Regular NN (Keras)
print("\nTraining Regular NN (Keras)...")
nn = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(784,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])
nn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = time.time()
nn.fit(x_train_flat, y_train, epochs=5, batch_size=32, verbose=0)
_, nn_acc = nn.evaluate(x_test_flat, y_test, verbose=0)
results['Regular NN (Keras)'] = (nn_acc, time.time()-start, 'Neural Net')
print(f"   Done! {nn_acc*100:.2f}%")

# 4. CNN (Keras) - THE STAR OF THIS WEEK!
print("\nTraining CNN (Keras)...")
cnn = keras.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = time.time()
cnn.fit(x_train_cnn, y_train, epochs=5, batch_size=64, verbose=0)
_, cnn_acc = cnn.evaluate(x_test_cnn, y_test, verbose=0)
results['CNN (Keras)'] = (cnn_acc, time.time()-start, 'CNN')
print(f"   Done! {cnn_acc*100:.2f}%")

# SCOREBOARD
print("\n" + "=" * 65)
print("FINAL SCOREBOARD - MNIST Digit Classification")
print("=" * 65)
print(f"{'Approach':<25} {'Accuracy':<12} {'Time':<10} {'Type'}")
print("-" * 65)
for name, (acc, t, tp) in sorted(results.items(), key=lambda x: -x[1][0]):
    medal = ">> " if acc == max(v[0] for v in results.values()) else "   "
    print(f"{medal}{name:<23} {acc*100:.2f}%       {t:.1f}s       {tp}")
print("-" * 65)
print("\nNotice how the CNN beats the regular NN - especially on images!")

### Record your results:

| Approach | Accuracy | Training Time | Type |
| --- | --- | --- | --- |
| Random Forest | ____% | ____s | Classic ML |
| Regular NN (scikit-learn) | ____% | ____s | Neural Net |
| Regular NN (Keras) | ____% | ____s | Neural Net |
| CNN (Keras) | ____% | ____s | CNN |